# Model evaluation to run0-spinup_steadystate

This example shows how to load model run0-spinup_steadystate data

In [ ]:
# # skip this if package has already been installed
# !pip install modvis
import modvis.ats_xdmf as xdmf
from modvis import colors

# [not used]
# from modvis import posixpath
# from modvis import ATSutils
# from modvis import utils
# from modvis import general_plots as gp

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
import h5py

model_dir = "./NF01"
cv_key = 'surface-cell_volume'

generate_plots = False

## Load model data

This will load the `water_balance.dat` file generated from ATS model. The data file includes watershed variables including outlet discharge, ET, and etc. By default, setting `plot=True` will show the water balance plots for `global, canopy, snow, surface, and subsurface` domains. **It is important to check if `max error` is close to zero!** Otherwise, there may be a water balance issue in the model.

In [ ]:
data_file_wb = os.path.join(model_dir, 'water_balance.dat')
data_wb = pd.read_csv(data_file_wb, comment='#')

In [ ]:
data_wb
# {right, left, bottom, top, front, back} face flux all have the same unit [mol d^-1]

In [ ]:
# the goal of vis/obs_file_time_interval here is to make comparable plot
# between results from different time interval setting
vis_file_time_interval = 0.1 # unit is day. convert the unit if in xml it's not configured as day
obs_file_time_interval = 1.0

# [to do] automatically determine time_interval from data_wb['time [d]']
# # Calculate cumulative flux by integrating over time
# # Convert time from days to seconds for integration
# time_days = data_wb['time [d]']
# dt_days = np.diff(time_days)
# dt_days = np.append(dt_days, dt_days[-1])  # Add last timestep

In [ ]:
# convert precipitation unit
# water_density = 1000  # kg/m³
# water_molar_mass = 0.018  # kg/mol
rho_m1 = 55500. # moles/m^3, water molar density
rho_m2 = 55000.
surface_area1 = 680 * 1  # m²; y=1 is determined in m2 = watershed_workflow.mesh.Mesh2D.from_Transect(x,z)

with h5py.File(os.path.join(model_dir, 'ats_vis_surface_data.h5'),'r') as d:
    a_key = list(d[cv_key].keys())[0]
    surface_area2 = d[cv_key][a_key][:].sum() # m^2
print(surface_area2)

# Conversion from m/s to mol/s
precipitation_m_per_d = data_wb['rain precipitation [m per time interval]'] / obs_file_time_interval
precipitation_mol_per_d = data_wb['rain precipitation [m per time interval]'] * 55000. * surface_area2

In [ ]:
# water_flux_left_face_m_per_d and water_flux_right_face_m_per_d
# unit m/d, interpretation: amount of water into the domain, in meter, assuming per surface_area2.
# in another word, viewing all flux (unit m/d) as influx/efflux of surface
water_flux_left_face_m_per_d = data_wb['left face flux [mol per time interval]'] / rho_m2 / surface_area2 / obs_file_time_interval
water_flux_right_face_m_per_d = data_wb['right face flux [mol per time interval]'] / rho_m2 / surface_area2 / obs_file_time_interval

# unit mol/d
# [note] vars are independent to surface area in this unit
water_flux_left_face_mol_per_d = data_wb['left face flux [mol per time interval]'] / obs_file_time_interval
water_flux_right_face_mol_per_d = data_wb['right face flux [mol per time interval]'] / obs_file_time_interval

# water balance check

In [ ]:
# water balance check
# noticing error here also depends on time step

# the water storage change
total_water_initial = (data_wb['surface water content [mol]'][0] + 
                      data_wb['subsurface water content [mol]'][0])
storage_change = ((data_wb['surface water content [mol]'] + 
                 data_wb['subsurface water content [mol]']) - total_water_initial) / rho_m2 / surface_area2

# the net flux (precipitation - runoff) in m/s
runoff_m_per_d = data_wb['runoff generation [mol per time interval]'] / rho_m2 / surface_area2 / obs_file_time_interval
runoff_mol_per_d = data_wb['runoff generation [mol per time interval]'] / obs_file_time_interval
net_flux = precipitation_m_per_d - runoff_m_per_d - water_flux_left_face_m_per_d - water_flux_right_face_m_per_d
net_flux_mol_per_d = precipitation_mol_per_d - runoff_mol_per_d - water_flux_left_face_mol_per_d - water_flux_right_face_mol_per_d

# cumulative_flux = np.cumsum(net_flux * dt_days)
#cumulative_water -> changes of water amount during 1 time interval
cumulative_flux = np.cumsum(net_flux)*obs_file_time_interval

In [ ]:
print(storage_change)
print(cumulative_flux)

In [ ]:
print(precipitation_mol_per_d)
print(runoff_mol_per_d)
print(water_flux_left_face_mol_per_d)
print(water_flux_right_face_mol_per_d)
print(net_flux_mol_per_d)

# at steady state, the value below becomes almost ZERO
print(precipitation_mol_per_d.iloc[-1]
      - water_flux_left_face_mol_per_d.iloc[-1]
      - runoff_mol_per_d.iloc[-1]
      - water_flux_right_face_mol_per_d.iloc[-1])

In [ ]:
# Create the figure with subplots
fig, ax = plt.subplots(2, 3, figsize=(12, 6))

# Plot 1: Subsurface Water Content
ax[0,0].plot(data_wb['time [d]'], data_wb['subsurface water content [mol]']/ rho_m2 / surface_area2, 
         label='Subsurface Water Content', linewidth=2)
ax[0,0].set_xlabel('Time [d]')
ax[0,0].set_ylabel('Subsurface Water Content [m]')
ax[0,0].set_title('Subsurface Water Content')
ax[0,0].grid(True, linestyle='--', alpha=0.7)
ax[0,0].legend()
ax[0,0].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# Plot 2: Surface Water Content
ax[0,1].plot(data_wb['time [d]'], data_wb['surface water content [mol]'] / rho_m2 / surface_area2, 
         label='Surface Water Content', linewidth=2)
ax[0,1].set_xlabel('Time [d]')
ax[0,1].set_ylabel('Surface Water Content [m]')
ax[0,1].set_title('Surface Water Content')
ax[0,1].grid(True, linestyle='--', alpha=0.7)
ax[0,1].legend()
ax[0,1].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# Plot 3: Precipitation
# ax[0,2].plot(data_wb['time [d]'], data_wb['precipitation [m d^-1]'], 
#          label='precipitation', linewidth=2)
ax[0,2].plot(data_wb['time [d]'], precipitation_m_per_d, 
         label='precipitation', linewidth=2)
ax[0,2].set_xlabel('Time [d]')
ax[0,2].set_ylabel('precipitation [m d^-1]')
ax[0,2].set_title('precipitation')
ax[0,2].grid(True, linestyle='--', alpha=0.7)
ax[0,2].legend()
ax[0,2].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# Plot 4: runoff generation
ax[1,0].plot(data_wb['time [d]'], runoff_m_per_d, 
         label='runoff generation', linewidth=2)
ax[1,0].set_xlabel('Time [d]')
ax[1,0].set_ylabel('runoff generation [m d^-1]')
ax[1,0].set_title('runoff generation')
ax[1,0].grid(True, linestyle='--', alpha=0.7)
ax[1,0].legend()
ax[1,0].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# # Plot 3: Groundwater Table
# data_file_gt = os.path.join(model_dir, 'groundwater_table.dat')
# data_gt = pd.read_csv(data_file_gt, comment='#')
# ax[1,1].plot(data_gt['time [d]'], data_gt['groundwater table'], 
#          label='Groundwater table', linewidth=2)
# ax[1,1].set_xlabel('Time [d]')
# ax[1,1].set_ylabel('Groundwater table [m]')
# ax[1,1].set_title('Groundwater Table')
# ax[1,1].grid(True, linestyle='--', alpha=0.7)
# ax[1,1].legend()
# ax[1,1].ticklabel_format(style='sci', axis='y', scilimits=(0,0))


ax[1,2].plot(data_wb['time [d]'], storage_change, 
         label='Storage Change', linewidth=2)
ax[1,2].plot(data_wb['time [d]'], cumulative_flux, 
         label='Cumulative Net Flux', linewidth=2)
ax[1,2].plot(data_wb['time [d]'], storage_change - cumulative_flux, 
         label='Water Balance Error', linewidth=2, linestyle='--')
ax[1,2].set_xlabel('Time [d]')
ax[1,2].set_ylabel('Water [m]')
ax[1,2].set_title('Water Balance')
ax[1,2].grid(True, linestyle='--', alpha=0.7)
ax[1,2].legend()
ax[1,2].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# Adjust layout
plt.tight_layout()
plt.show()

# read visualization file

In [ ]:
# for [subsurface right face], compare darcy_velocity from visualization file and water_flux from observation file
# for [surface right face], it's not comparable. I don't understand surface-velocity.

# based on ATS online documentation, unit of darcy_velocity is m/s, while unit of water_flux is mol/s
# unit conversion: darcy_velocity*surface_area*55000 mol/m3
# noticing: observation file is doing [extensive integral] of water_flux

In [ ]:
xlim_preset = (-10,690)
ylim_preset = (940,1120)

In [ ]:
vis_surf = xdmf.VisFile(directory=model_dir, prefix='ats_vis',
                            domain="surface", 
                            # filename="ats_vis_surface_data.h5" , 
                            # mesh_filename="ats_vis_surface_mesh.h5", 
                            model_time_unit='d')
#select same output as subsurface
#vis_surf.filterIndices(steps)
vis_surf.loadMesh(order=['x','z'])

vis = xdmf.VisFile(directory=model_dir, prefix='ats_vis',
                       # filename="ats_vis_data.h5", 
                       # mesh_filename="ats_vis_mesh.h5", 
                       model_time_unit='d')
#select output every 2 days
#vis.filterIndices(steps)
vis.loadMeshPolygons()

In [ ]:
# Plot
fig, ax = plt.subplots(1,1,figsize=(10,2.6),sharex=True)
# norm = mcolors.TwoSlopeNorm(vmin=-16, vcenter=-15, vmax=-11.5)
sat = vis.get("permeability", vis.cycles[0]); sat = np.log10(sat)
poly = vis.getMeshPolygons(cmap='cividis', linewidth=0.1, edgecolor='k')
# sat = vis.get("saturation_liquid", vis.cycles[0]); sat[:] = 0
# poly = vis.getMeshPolygons(cmap='Blues_r', linewidth=0.1, edgecolor='k', norm=norm)
poly.set_array(sat)
poly.set_clim(-16,-11.5)
ax.add_collection(poly)

elev = vis_surf.get('surface-elevation', vis.cycles[-1])
depth = vis_surf.get('surface-ponded_depth', vis.cycles[-1])

ax.plot(vis_surf.centroids[:,0], elev+depth, 'white', linewidth=2, alpha=0)
# ax.axhline(y=0)
plt.colorbar(poly,shrink=1., ax=ax, label = r'$\text{log} \ \text{Permeability} \ (m^2)$')
# ax.plot(0,0)
ax.set(xlabel='Distance (m)', ylabel='Elevation (m)',ylim=ylim_preset, xlim=xlim_preset)

if generate_plots:
    output_filename = '../images/fig1c-hillslope2d.tif'
    fig.savefig(output_filename, dpi=300, pil_kwargs={'compression': 'tiff_lzw'})

In [ ]:
# from full_watershed-workflow
# 18 layers in z
dzs_soil = [0.05, 0.05, 0.05, 0.1, 0.25, 0.5, 0.5, 0.5]
dzs_geo = [1., 1., 1.5, 1.5, 2., 2., 2., 3., 3., 3.]
dzs_merged = np.concatenate((dzs_soil, dzs_geo))

print(dzs_merged)

# since width in y-axis is 1m, so areas = dzs_merged

In [ ]:
# Assuming your x-coordinates increase from left to right
# Find the maximum x-coordinate of the cell centroids
max_x = np.max(vis.centroids[:, 0])

# Identify the indices of the cells whose centroids have an x-coordinate
# very close to the maximum x-coordinate. This will give you the cells
# on the right boundary. You might need to adjust the tolerance
# depending on the precision of your mesh.
tolerance = 1e-6  # Adjust as needed
right_boundary_cell_indices = np.where(np.abs(vis.centroids[:, 0] - max_x) < tolerance)[0]

# Sort the right boundary cell indices based on their z-centroid
# from largest (top) to smallest (bottom)
sorted_indices = right_boundary_cell_indices[np.argsort(vis.centroids[right_boundary_cell_indices, 2])[::-1]]

print("Original right boundary cell indices:", right_boundary_cell_indices)
print("Sorted right boundary cell indices (top to bottom):", sorted_indices)
print(vis.centroids[sorted_indices, 2])

## based on darcy_velocity.0 of right boundary CELLS

In [ ]:
varn = "darcy_velocity.0"
all_steps_sum = []

for step in range(len(vis.cycles)):
    zdata = vis.get(varn, vis.cycles[step])

    # Multiply the zdata at the sorted right boundary indices by the merged array
    # Ensure that the lengths are compatible for element-wise multiplication
    if len(zdata[sorted_indices]) == len(dzs_merged):
        multiplied_data = zdata[sorted_indices] * dzs_merged
        step_sum = np.sum(multiplied_data)
        all_steps_sum.append(step_sum)
    else:
        print(f"Warning: Length mismatch between zdata[sorted_indices] ({len(zdata[sorted_indices])}) and merged_array ({len(merged_array)}). Skipping multiplication and summation for this step.")
        all_steps_sum.append(None) # Or some other indicator of failure

#print("\nSum of multiplied data for all steps:", all_steps_sum)

all_steps_sum_array = np.array(all_steps_sum)

In [ ]:
print(dir(vis))
print(vis.times)

In [ ]:
# all_steps_sum is in m3/s
# compare with data_wb['right face flux'] in mol/(time interval)

xmin = -10
xmax = 3660

# Create the figure with subplots
fig, ax = plt.subplots(1, 3, figsize=(12, 3))

# Get the time data for coloring
time_data_obs = data_wb['time [d]']
time_data_vis = vis.times

# Plot 1: water_flux from observation file
ax[0].scatter(time_data_obs, data_wb['right face flux [mol per time interval]']/obs_file_time_interval, 
           c=time_data_obs, cmap='viridis', s=20)
ax[0].set_xlabel('Time [d]')
ax[0].set_ylabel('water flux [mol/d]')
ax[0].grid(True, linestyle='--', alpha=0.7)
ax[0].ticklabel_format(style='sci', axis='y', scilimits=(0,0))
ax[0].set_xlim(xmin, xmax)
ax[0].set_title('Obs - right face flux')

# Plot 2: flux based on darcy_velocity from visualization file
ax[1].scatter(time_data_vis, all_steps_sum_array*rho_m2*86400, 
           c=time_data_vis, cmap='viridis', s=20)
ax[1].set_xlabel('Time [d]')
ax[1].set_ylabel('water flux [mol/d]')
ax[1].grid(True, linestyle='--', alpha=0.7)
ax[1].ticklabel_format(style='sci', axis='y', scilimits=(0,0))
ax[1].set_xlim(xmin, xmax)
ax[1].set_title('Vis - darcy flux')

# Plot 3: 1-1 plot
if vis_file_time_interval==obs_file_time_interval:
    ax[2].scatter(data_wb['right face flux [mol per time interval]']/obs_file_time_interval, all_steps_sum_array*rho_m2*86400,
                  c=time_data_obs, cmap='viridis', marker='.', s=50)
    ax[2].set_xlabel('Obs - water flux')
    ax[2].set_ylabel('Vis - darcy velocity * molar density')
    ax[2].ticklabel_format(style='sci', axis='x', scilimits=(0,0))
    ax[2].ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    x_vals = np.linspace(0, 1.3e5, 100)
    ax[2].plot(x_vals, x_vals, 'k--')
    ax[2].set_title('Vis - Obs 1-1 Plot')
    tmp_ratio = (all_steps_sum_array*rho_m2*86400)/(data_wb['right face flux [mol per time interval]']/obs_file_time_interval)
    print("ratio of darcy_vel based water flux vs water flux from vis file: ", tmp_ratio.iloc[-1])

# tracer basic plots

In [ ]:
# obtain water_flux from Visualization, and compare with Observation

In [ ]:
# [to do] 250522
# with the update of "1-full_workflow_OakCreek.ipynb"
# rightx/righty/22.0 below can be loaded from "../data-processed/SF01/m2_SF01_nx100.mat"

# topx and topy are from 1-full_workflow_OakCreek.ipynb
rightx = np.array([  0. ,   6.8,  13.6,  20.4,  27.2,  34. ,  40.8,  47.6,  54.4,  61.2,
  68. ,  74.8,  81.6,  88.4,  95.2, 102. , 108.8, 115.6, 122.4, 129.2,
 136. , 142.8, 149.6, 156.4, 163.2, 170. , 176.8, 183.6, 190.4, 197.2,
 204. , 210.8, 217.6, 224.4, 231.2, 238. , 244.8, 251.6, 258.4, 265.2,
 272. , 278.8, 285.6, 292.4, 299.2, 306. , 312.8, 319.6, 326.4, 333.2,
 340. , 346.8, 353.6, 360.4, 367.2, 374. , 380.8, 387.6, 394.4, 401.2,
 408. , 414.8, 421.6, 428.4, 435.2, 442. , 448.8, 455.6, 462.4, 469.2,
 476. , 482.8, 489.6, 496.4, 503.2, 510. , 516.8, 523.6, 530.4, 537.2,
 544. , 550.8, 557.6, 564.4, 571.2, 578. , 584.8, 591.6, 598.4, 605.2,
 612. , 618.8, 625.6, 632.4, 639.2, 646. , 652.8, 659.6, 666.4, 673.2,
 680. ])
righty = np.array([1102.011, 1101.917, 1101.917, 1101.917, 1099.421, 1099.421, 1098.469,
 1097.036, 1090.794, 1080.269, 1077.915, 1075.28 , 1072.323, 1072.323,
 1065.303, 1061.85 , 1057.017, 1055.526, 1053.547, 1053.547, 1053.547,
 1053.547, 1053.547, 1053.547, 1053.547, 1053.547, 1053.547, 1051.923,
 1051.923, 1051.923, 1051.923, 1051.923, 1051.499, 1051.009, 1048.841,
 1048.111, 1046.253, 1043.301, 1042.374, 1041.46 , 1041.46 , 1039.801,
 1039.101, 1036.813, 1036.478, 1035.245, 1034.981, 1033.532, 1033.532,
 1033.099, 1032.963, 1032.855, 1032.711, 1031.332, 1030.189, 1030.11 ,
 1029.083, 1029.083, 1028.996, 1028.902, 1028.769, 1028.612, 1027.619,
 1026.707, 1026.257, 1026.036, 1026.036, 1024.497, 1024.008, 1023.479,
 1022.879, 1022.248, 1017.741, 1016.066, 1015.267, 1015.267, 1012.226,
 1011.418, 1010.598, 1009.746, 1004.037, 1001.763, 1001.763, 1001.763,
 1001.763, 1001.763,  999.904,  999.904,  999.476,  993.245,  992.59 ,
  991.7  ,  991.7  ,  990.476,  988.979,  987.216,  983.303,  981.041,
  976.897,  972.046,  972.046])
leftx=rightx
lefty=righty-22.0 # due toin 1-full_workflow_OakCreek.ipynb, extrude 22m in depth

In [ ]:
# plot Flow Path along the 2D Transect
import matplotlib.pyplot as plt
from matplotlib import path as mpath
from matplotlib import patches as mpatches
import numpy as np
import matplotlib.colors as colors

from matplotlib.ticker import FixedLocator,FormatStrFormatter

# norm = colors.TwoSlopeNorm(vmin=-30, vcenter=0, vmax=160)
x=(vis.centroids)[:,0]
z=(vis.centroids)[:,2]

def create_tricontour_plot(step, vis, cont=None):
    # Create a new figure and axis for the current step
    fig, ax = plt.subplots(figsize=(5,2.6), sharey=True)
    sat = vis.get("saturation_liquid", vis.cycles[step])
    poly = vis.getMeshPolygons(cmap='bwr_r', linewidth=0, edgecolor='none')
    poly.set_array(sat)
    poly.set_clim(0.0,1.0)
    ax.add_collection(poly)

    cbar=fig.colorbar(poly,shrink=1., ax=ax, label = 'Saturation', pad=0.02)
    # Set the tick locations to the minimum and maximum values
    cbar.ax.yaxis.set_major_locator(FixedLocator([0,0.25,0.50,0.75, 1.0]))
    # Format the tick labels with sufficient decimal places
    cbar.ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f')) # Adjust as needed
    # Optionally, you can remove intermediate ticks if any are present
    cbar.ax.yaxis.set_minor_locator(FixedLocator([]))
    # Control colorbar tick label font size
    cbar.ax.tick_params(labelsize=10) # Adjust 10 as needed
    
    ax.set_xlim(xlim_preset)
    ax.set_ylim(ylim_preset)

    ax.tick_params(axis='both', which='major', labelsize=10) # Adjust 10 as needed

    # a line for water table
    elev = vis_surf.get('surface-elevation', vis.cycles[i])
    pd = vis_surf.get('surface-ponded_depth', vis.cycles[i])
    wt = vis_surf.get('surface-water_table_depth', vis.cycles[i])
    axpd = ax.plot(vis_surf.centroids[:,0], elev+pd-wt, 'b', linewidth=1.5)
    
    v1 = vis.get('darcy_velocity.0', vis.cycles[i])
    v3 = vis.get('darcy_velocity.2', vis.cycles[i])
    stride = 8
    x_quiver = x[::stride]
    z_quiver = z[::stride]
    v1_quiver = v1[::stride]
    v3_quiver = v3[::stride]
    #axes.quiver(x, z, v1, v3, width=0.0008,color='black')
    ax.quiver(x_quiver, z_quiver, v1_quiver, v3_quiver, width=0.002, color='black')
    
    # Return the plot
    return fig, ax, axpd, cont


cont = None  # Define cont object outside the loop

# Loop over each step and create a tricontour plot
for i, step in enumerate([3650]):
    # Call the create_tricontour_plot function and save the cont object
    fig, axes, axpd, cont = create_tricontour_plot(i, vis, cont=cont)
    # Add labels to the subplot

fig.text(0.45, 0.03, 'Distance [m]', fontsize=12, ha='center')
fig.text(0.03, 0.35, 'Elevation [m]', fontsize=12, ha='center', rotation=90)

plt.tight_layout(rect=[0.03, 0.05, 1, 1]) # Adjust bottom padding if needed

generate_plots=False
if generate_plots:
    output_filename = '../images/fig2a-darcyvel-v2-2.tif'
    fig.savefig(output_filename, dpi=300, pil_kwargs={'compression': 'tiff_lzw'})
    #output_filename = '../images/fig2a-darcyvel.svg'
    #fig.savefig(output_filename)

In [ ]:
print(step)
step=1
print(vis.cycles[step])

In [ ]:
print(len(vis.cycles))

In [ ]:
i=0
#step=1
varn="total_component_concentration.CH2O(aq)"

xdata = vis.centroids[:, 0]
ydata = vis.centroids[:, 2]
zdata = vis.get(varn, vis.cycles[step])
print(zdata)
print([np.min(zdata), np.max(zdata)])

zlim_preset = (7e-7, 51e-7)

In [ ]:
all_local_mins = []
all_local_maxs = []

for step in range(len(vis.cycles)):
    zdata = vis.get(varn, vis.cycles[step])
    local_min = np.min(zdata)
    local_max = np.max(zdata)

    all_local_mins.append(local_min)
    all_local_maxs.append(local_max)

global_min = np.min(all_local_mins)
global_max = np.max(all_local_maxs)
print("--- Summary ---")
#print(f"Array of all Local Minimums:\n{all_local_mins}")
#print(f"Array of all Local Maximums:\n{all_local_maxs}")
print(f"Global Minimum (np.min(all_local_mins)): {global_min}")
print(f"Global Maximum (np.max(all_local_maxs)): {global_max}")

zlim_preset = (global_min, global_max)

In [ ]:
# plot the tracer1 concentration
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from matplotlib import path as mpath
from matplotlib import patches as mpatches
import numpy as np

import sys, os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.colors as mcolors

from matplotlib.ticker import FixedLocator,FormatStrFormatter
# Plotting parameters
# rc('text', usetex=False)
small_size = 10
medium_size = 25
bigger_size = 30
plt.rc('font', size=small_size)          # controls default text sizes
plt.rc('axes', titlesize=small_size)    # fontsize of the axes title
plt.rc('axes', labelsize=small_size)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=small_size)    # fontsize of the tick labels
plt.rc('ytick', labelsize=small_size)    # fontsize of the tick labels
plt.rc('legend', fontsize=small_size)    # legend fontsize
plt.rc('figure', titlesize=bigger_size)  # fontsize of the figure title
plt.rc('text', usetex = False)

# Create a new figure with three subfigures
fig, axes = plt.subplots(4, 1, figsize=(10, 10.4))

def create_tricontour_plot(step, vis,varn, log, clim, 
             cmap='Red', xlim=None, ylim=None, showtime=True, ax=None):
    # Get the x, y, and z data for the tricontour plot
    xdata = vis.centroids[:, 0]
    ydata = vis.centroids[:, 2]
    zdata = vis.get(varn, vis.cycles[step])# can do unit conversion here, *32*1000*55
    if log: zdata = np.log10(zdata)
    #if np.any(zdata>7.5):
        # Add a small positive value to zdata to avoid zeros or negative values
     #   zdata[zdata>7.5] = 7.5

    vertices = np.vstack([np.concatenate([leftx, rightx[::-1]]), np.concatenate([lefty, righty[::-1]])]).T
    poly_codes = [mpath.Path.MOVETO] + (len(vertices) - 1) * [mpath.Path.LINETO]
    path = mpath.Path(vertices, poly_codes)
    clip_patch = mpatches.PathPatch(path, facecolor='none', edgecolor='none')
    ax.add_patch(clip_patch)

    # Create the tricontour plot with the current clipping patch
    #bounds=np.linspace(0,13,100)  # the user can changed the last number to get different contour like from 10 to 100
    bounds=np.linspace(zlim_preset[0],zlim_preset[1],100)
    cont = ax.tricontourf(xdata, ydata, zdata, vmax=zlim_preset[1], vmin=zlim_preset[0],levels=bounds, cmap=cmap)

    for col in cont.collections:
        col.set_clip_path(clip_patch.get_path(), clip_patch.get_transform())
    

    #cont.set_clim(*clim) # set colorbar limits
    cbar=fig.colorbar(cont, ax=ax, label='Concentration (molS/molH2O)') # mol_tracer per mol_water
    #cbar.ax.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))  # Set the decimal places
    # Set the tick locations to the minimum and maximum values
    #cbar.ax.yaxis.set_major_locator(FixedLocator([zlim_preset[0],0.25,0.50,0.75, zlim_preset[1]]))
    tick_locs = np.linspace(zlim_preset[0], zlim_preset[1], 5)
    cbar.ax.yaxis.set_major_locator(FixedLocator(tick_locs))
    # Format the tick labels with sufficient decimal places
    cbar.ax.yaxis.set_major_formatter(FormatStrFormatter('%.3e')) # Adjust as needed
    # Optionally, you can remove intermediate ticks if any are present
    #cbar.ax.yaxis.set_minor_locator(FixedLocator([0.25,0.50,0.75]))

    ax.set_xlim(xlim_preset)
    ax.set_ylim(ylim_preset)

    elev = vis_surf.get('surface-elevation', vis.cycles[i])
    pd = vis_surf.get('surface-ponded_depth', vis.cycles[i])
    wt = vis_surf.get('surface-water_table_depth', vis.cycles[i])
    pd_smooth = savgol_filter(pd, window_length=11, polyorder=2)
    wt_smooth = savgol_filter(wt, window_length=11, polyorder=2)
    axpd = ax.plot(vis_surf.centroids[:,0], elev+pd_smooth-wt_smooth, 'black', linewidth=2)

    v1 = vis.get('darcy_velocity.0', vis.cycles[i])
    v3 = vis.get('darcy_velocity.2', vis.cycles[i])
    #axes.quiver(x, z, v1, v3, width=0.0008,color='w')
    
    if showtime:
        # days = step // 24  # calculate the number of days
        # time = step % 24  # calculate the time within a day
        # time_str = f'{time:02d}:00'  # format the time as 'hh:00'
        # ax.text(0.8, 0.95, 'Day {} {}'.format(days, time_str), transform=ax.transAxes, fontsize=20,
        #         fontweight='bold', va='top', ha='left')
        days = step
        ax.text(0.8, 0.95, 'Day {}'.format(days), transform=ax.transAxes, fontsize=12,
                fontweight='bold', va='top', ha='left')

# Loop over each step and create a tricontour plot in each subfigure
for i, step in enumerate([0,400,800,1200]): #([0,1210,2420,3650]):
    create_tricontour_plot(step, vis, varn=varn, log=False,
                            clim=[0,100], cmap='rainbow', ax=axes[i])

# Add labels and colorbar to the plot
fig.suptitle(f"Temporal Variation of {varn} Across Different Days", fontsize=12, x=0.45, y=0.925)
# fig.text(0.635, 0.835,'Water level')
# fig.text(0.635, 0.635,'Water level')
# fig.text(0.635, 0.435,'Water level')
# fig.text(0.635, 0.235,'Water level')

fig.text(0.45, 0.06, 'Distance [m]', fontsize=12, ha='center')
fig.text(0.05, 0.5, 'Elevation [m]', fontsize=12, ha='center', rotation=90)
#fig.colorbar(im, ax=axes.ravel().tolist(), label='[m/s]')
#plt.savefig('Temporal_dynamic_oxygen_o2_case_et.png',dpi=300) 


In [ ]:
def create_tricontour_plot(step, vis,varn, log, clim, 
             cmap='Red', xlim=None, ylim=None, showtime=True, ax=None):
    # Get the x, y, and z data for the tricontour plot
    xdata = vis.centroids[:, 0]
    ydata = vis.centroids[:, 2]
    zdata = vis.get(varn, vis.cycles[step])# can do unit conversion here, *32*1000*55
    if log: zdata = np.log10(zdata)
    #if np.any(zdata>7.5):
        # Add a small positive value to zdata to avoid zeros or negative values
     #   zdata[zdata>7.5] = 7.5

    vertices = np.vstack([np.concatenate([leftx, rightx[::-1]]), np.concatenate([lefty, righty[::-1]])]).T
    poly_codes = [mpath.Path.MOVETO] + (len(vertices) - 1) * [mpath.Path.LINETO]
    path = mpath.Path(vertices, poly_codes)
    clip_patch = mpatches.PathPatch(path, facecolor='none', edgecolor='none')
    ax.add_patch(clip_patch)

    # Create the tricontour plot with the current clipping patch
    #bounds=np.linspace(0,13,100)  # the user can changed the last number to get different contour like from 10 to 100
    bounds=np.linspace(zlim_preset[0],zlim_preset[1],100)
    cont = ax.tricontourf(xdata, ydata, zdata, vmax=zlim_preset[1], vmin=zlim_preset[0],levels=bounds, cmap=cmap)

    for col in cont.collections:
        col.set_clip_path(clip_patch.get_path(), clip_patch.get_transform())
    

    #cont.set_clim(*clim) # set colorbar limits
    cbar=fig.colorbar(cont, ax=ax, label='Concentration (molS/molH2O)', pad=0.02)
    #cbar.ax.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))  # Set the decimal places
    # Set the tick locations to the minimum and maximum values
    #cbar.ax.yaxis.set_major_locator(FixedLocator([zlim_preset[0],0.25,0.50,0.75, zlim_preset[1]]))
    tick_locs = np.linspace(zlim_preset[0], zlim_preset[1], 5)
    cbar.ax.yaxis.set_major_locator(FixedLocator(tick_locs))
    # Format the tick labels with sufficient decimal places
    cbar.ax.yaxis.set_major_formatter(FormatStrFormatter('%.2e')) # Adjust as needed
    # Optionally, you can remove intermediate ticks if any are present
    #cbar.ax.yaxis.set_minor_locator(FixedLocator([0.25,0.50,0.75]))
    # Control colorbar tick label font size
    cbar.ax.tick_params(labelsize=10) # Adjust 10 as needed

    #fig.colorbar(axes[0].collections[0], ax=axes, label='[Log(mol/L)]')
    ax.set_xlim(xlim_preset)
    ax.set_ylim(ylim_preset)

    ax.tick_params(axis='both', which='major', labelsize=10) # Adjust 10 as needed

    elev = vis_surf.get('surface-elevation', vis.cycles[i])
    pd = vis_surf.get('surface-ponded_depth', vis.cycles[i])
    wt = vis_surf.get('surface-water_table_depth', vis.cycles[i])
    pd_smooth = savgol_filter(pd, window_length=11, polyorder=2)
    wt_smooth = savgol_filter(wt, window_length=11, polyorder=2)
    axpd = ax.plot(vis_surf.centroids[:,0], elev+pd_smooth-wt_smooth, 'black', linewidth=1.0)

    v1 = vis.get('darcy_velocity.0', vis.cycles[i])
    v3 = vis.get('darcy_velocity.2', vis.cycles[i])
    #axes.quiver(x, z, v1, v3, width=0.0008,color='w')
    
    if showtime:
        # days = step // 24  # calculate the number of days
        # time = step % 24  # calculate the time within a day
        # time_str = f'{time:02d}:00'  # format the time as 'hh:00'
        # ax.text(0.8, 0.95, 'Day {} {}'.format(days, time_str), transform=ax.transAxes, fontsize=20,
        #         fontweight='bold', va='top', ha='left')
        days = step
        ax.text(0.8, 0.95, 'Day {}'.format(days), transform=ax.transAxes, fontsize=12,
                fontweight='bold', va='top', ha='left')


fig, axes = plt.subplots(1,1,figsize=(5,2.6))
create_tricontour_plot(step, vis, varn="total_component_concentration.CH2O(aq)", log=False,
                        clim=[0,100], cmap='rainbow', showtime=False, ax=axes)

fig.text(0.45, 0.03, 'Distance [m]', fontsize=10, ha='center')
fig.text(0.03, 0.35, 'Elevation [m]', fontsize=10, ha='center', rotation=90)

plt.tight_layout(rect=[0.03, 0.05, 1, 1]) # Adjust bottom padding if needed

#generate_plots=True
if generate_plots:
    output_filename = '../images/fig2b-massdistribution-v2.tif'
    fig.savefig(output_filename, dpi=300, pil_kwargs={'compression': 'tiff_lzw'})
    #output_filename = '../images/fig2a-darcyvel.svg'
    #fig.savefig(output_filename)

In [ ]:
# --- Create output directory if it doesn't exist ---
output_folder = './images'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Created directory: {output_folder}")
else:
    print(f"Directory already exists: {output_folder}")

# --- Initialize frame counter ---
frame_counter = 0
# --- Loop through steps and save figures ---
for step in range(0, 1664,1): # Loop from 0 up to and including 365
    print(f"Generating figure for step {step}...")

    # Ensure the step is a valid index for vis.times
    if step >= len(vis.times):
        print(f"Warning: step {step} is out of bounds for vis.times. Skipping.")
        continue

    vis_time_value = vis.times[step]/86400

    fig, axes = plt.subplots(1,1,figsize=(5,2.6))
    create_tricontour_plot(step, vis, varn="total_component_concentration.CH2O(aq)", log=False,
                            clim=[0,100], cmap='rainbow', showtime=False, ax=axes)
    
    axes.set_title(f"Time: {vis_time_value:.1f} d")
    fig.text(0.45, 0.03, 'Distance [m]', fontsize=10, ha='center')
    fig.text(0.03, 0.35, 'Elevation [m]', fontsize=10, ha='center', rotation=90)
    
    plt.tight_layout(rect=[0.03, 0.05, 1, 1]) # Adjust bottom padding if needed
    
    # Save the figure
    # Use zero-padding for the filename to ensure correct sorting for animation
    filename = os.path.join(output_folder, f'frame_{frame_counter:04d}.png') # e.g., frame_0000.png, frame_0001.png
    plt.savefig(filename, dpi=150, bbox_inches='tight') # dpi can be adjusted for quality
    plt.close(fig) # Close the figure to free up memory
    frame_counter += 1

print(f"Generated {step + 1} figures in '{output_folder}' directory.")
print("You can now use a tool like FFmpeg to create an animation:")
print(f"ffmpeg -framerate 10 -i {output_folder}/frame_%04d.png -c:v libx264 -r 30 -pix_fmt yuv420p -vf \"scale=1980:642\" output_animation.mp4")